# Week 1 - EDA - Yiscah Mark

### this notebook goes through basic cleaning and summaries of the eda_data.csv

###### Installing and importing necessary libraries and tools

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
%matplotlib inline
!pip install ydata_profiling


Defaulting to user installation because normal site-packages is not writeable
Looking in links: /usr/share/pip-wheels


In [10]:
from ydata_profiling import ProfileReport


#### Since I use a web version of Jupyter, I needed to load the file via its URL address. In the next cell, I print the first five rows to get a picture of the data.

In [12]:
# 1. Ingestion via URL
url = 'https://raw.githubusercontent.com/bellevue-university/dsc355/refs/heads/main/10%20Week/week_1/eda_data.csv'
df = pd.read_csv(url)

In [13]:
print(df.head())

          x0         x1         x2        x3         x4         x5  \
0 -17.933519   6.559220 -14.452810 -4.732855   0.381673   2.563194   
1 -37.214754  10.774930 -15.384004 -0.077339  10.983774 -15.210206   
2   0.330441 -19.609972  -9.167911  2.064124  12.071688  12.506141   
3 -13.709765  -8.011390   6.759264  1.727615  -1.768382  24.039733   
4  -4.202598   7.076210 -26.004919 -4.269696  -3.414224   2.115989   

            x6          x7         x8         x9     x10       x11        x12  \
0  ($1,306.52)  -89.394348 -28.454044 -16.201298  -0.01%  0.217010   9.729891   
1     ($24.86)  153.032652 -32.557736  69.675903   0.00% -3.584908  35.727926   
2    ($110.85) -141.437276 -20.794952  55.042604   0.00% -3.991366  -9.283523   
3    ($324.43)   51.039653  -7.046908 -31.424419   0.01%  7.908897  -2.891882   
4   $1,213.37   -31.046700  19.061182 -31.525515  -0.01%  0.846719  25.497480   

        x13         y  
0 -0.786431  0.666146  
1 -0.985552  0.378411  
2 -3.394718  0.62449

#### Observation: I don't know what this data is for, but it seems like y is the target. It may detect the probability of churn, or some other percentage, since the values are all between 0 and 1. The x6 columns look a bit messy; there are parentheses and dollar signs. The X10 column, as well, has a percent symbol. The first step in cleaning I did was to get rid of the parentheses and sign in x6 as well as change the percent value to a decimal, and remove the symbol.

In [14]:

# 2. Data Cleaning (Step-by-Step)

# Clean x6: Handle currency symbols, commas, and negative parentheses
def clean_currency(value):
    if isinstance(value, str):
        value = value.replace('$', '').replace(',', '')
        if '(' in value:
            value = '-' + value.replace('(', '').replace(')', '')
        return float(value)
    return value

df['x6'] = df['x6'].apply(clean_currency)


In [15]:
# Clean x10: Remove percentage sign and convert to decimal
df['x10'] = df['x10'].str.replace('%', '').astype(float) / 100



#### This follows the requirement of a basic statistical outline, a summary of the shape, and the data type in each column, as well as a numerical summary of each column. The summary includes the mean, max, min, standard deviation, percentile, and count.

In [16]:

print("Dataset Shape:", df.shape)
print("\nColumn Data Types:\n", df.dtypes)
print("\nNumerical Summary:\n", df.describe())



Dataset Shape: (9999, 15)

Column Data Types:
 x0     float64
x1     float64
x2     float64
x3     float64
x4     float64
x5     float64
x6     float64
x7     float64
x8     float64
x9     float64
x10    float64
x11    float64
x12    float64
x13    float64
y      float64
dtype: object

Numerical Summary:
                 x0           x1           x2           x3           x4  \
count  9996.000000  9995.000000  9996.000000  9997.000000  9997.000000   
mean      6.501091    -3.729880    -7.335819    -0.001323     1.347141   
std      29.140034    17.237178    38.355015     3.995307     9.606695   
min    -106.809919   -65.137848  -150.846091   -14.616540   -37.499530   
25%     -13.094564   -15.356197   -33.079854    -2.681308    -5.047927   
50%       6.659263    -3.825630    -7.374468     0.010637     1.217076   
75%      26.214107     7.764036    18.477979     2.635699     7.807128   
max     114.823451    67.685933   127.204103    16.923269    38.624213   

                x5        

#### The following is a more advanced summary of the data using ProfilerReport.

In [17]:
# 4. Generate Pandas Profiling Report

profile = ProfileReport(df, title="DSC355 EDA Data Report", explorative=True)

#### i saved the report to a new html file.

In [18]:
# Save the report
profile.to_file("eda_data_report.html")

Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]


100%|██████████| 15/15 [00:00<00:00, 27.89it/s]


Generate report structure:   0%|          | 0/1 [00:00<?, ?it/s]

Render HTML:   0%|          | 0/1 [00:00<?, ?it/s]

Export report to file:   0%|          | 0/1 [00:00<?, ?it/s]

#### I used this to display the file with the new very detailed report.

In [19]:
from IPython.display import IFrame
IFrame(src='./eda_data_report.html', width='100%', height=800)

### summary
#### The first thing I noticed was that 5 of them were not an issue. 4 columns were flagged as 100% unique. This makes a lot of sense because the values are all decimals with many spaces. One column got flagged for having 39% zeros. Because the columns aren't labeled, it is hard to know whether it is a mistake or intentional. I looked closely at the summary and noticed that the range was between -.0004 and .0004, 0.0 is smack in the middle, and it looks like it fits nicely into the column.
#### Another piece that caught my attention is the missing values. Out of such a big data set, a few missing values in each column shouldn't change the results so drastically. But I ended up replacing each missing value with the column's mean, then reran the report. 

In [20]:

# 3. Impute Missing Values (The "New" Step)
df = df.fillna(df.median(numeric_only=True))

# 4. Verify Cleanup
print(f"Total Missing Values after cleaning: {df.isnull().sum().sum()}")

# 5. Redid the Report
profile = ProfileReport(df, title="DSC355 - Fully Cleaned EDA Report", explorative=True)
profile.to_file("eda_data_report_v2.html")

Total Missing Values after cleaning: 0


Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]


100%|██████████| 15/15 [00:00<00:00, 23.01it/s]


Generate report structure:   0%|          | 0/1 [00:00<?, ?it/s]

Render HTML:   0%|          | 0/1 [00:00<?, ?it/s]

Export report to file:   0%|          | 0/1 [00:00<?, ?it/s]

In [22]:
from IPython.display import IFrame
IFrame(src='./eda_data_report_v2.html', width='100%', height=800)

#### At first, I was worried because there were more warnings than last time, but then I realized they were for unique values. In the interaction chart, you can view the correlation between any two variables. All the variables against y appear as a column of blue in the middle. When two x variables are viewed against each other, it's more like a ball in the middle. There are no more missing values.
#### EDA is a way to view and familiarize yourself with the data. It is important to gather all this information to use the data correctly and accurately.
